In [ ]:
# === CNN with 70:15:15 split, early stopping check, and training curves ===
import pandas as pd
import numpy as np
import time
import psutil
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from scipy.signal import savgol_filter
import shap

# === Memory usage ===
def get_memory_usage():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        process = psutil.Process()
        return process.memory_info().rss / (1024 ** 2)

# === SNV normalization ===
def snv(input_data):
    return (input_data - np.mean(input_data, axis=1, keepdims=True)) / np.std(input_data, axis=1, keepdims=True)

# === Step 1: Load and preprocess ===
df = pd.read_csv('Dataset _Raman.csv')
X = df.drop(columns=['Algae_Type', 'ID'])
y = df['Algae_Type']

X_sg = savgol_filter(X.values, window_length=25, polyorder=2, deriv=1, axis=1)
X_snv = snv(X_sg)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_snv)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# === Define CNN Model ===
class Raman1DCNN(nn.Module):
    def __init__(self, input_length, num_classes):
        super(Raman1DCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2)
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        self.flatten_dim = (input_length // 4) * 64
        self.fc1 = nn.Linear(self.flatten_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

        self.dropout = nn.Dropout(0.43870)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# === Train & Evaluate ===
def train_evaluate():
    # --- 70:15:15 split ---
    X_train, X_temp, y_train, y_temp = train_test_split(
        X_scaled, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    # Convert to tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)
    y_val_tensor = torch.tensor(y_val, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=32)
    test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32)

    # Init model
    input_length = X.shape[1]
    num_classes = len(np.unique(y_encoded))
    model = Raman1DCNN(input_length, num_classes)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.00057665)

    # Training setup
    epochs = 50
    best_val_loss = np.inf
    best_epoch = 0
    history = []

    start_train = time.time()
    for epoch in range(epochs):
        # ---- Training ----
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # ---- Validation loss for early stopping check ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1

        history.append({"Epoch": epoch+1, "Train Loss": train_loss, "Train Accuracy": train_acc})
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}")

    end_train = time.time()
    print(f"\n⚠️ Early stopping would have occurred at epoch {best_epoch} (lowest val loss).")

    # Save training curves
    df_history = pd.DataFrame(history)
    df_history.to_excel("cnn_training_curves.xlsx", index=False, engine="openpyxl")
    print("📊 Training curves saved to cnn_training_curves.xlsx")

    # === Test Evaluation ===
    start_test = time.time()
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())
    end_test = time.time()

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

    result = {
        'Model': "CNN",
        'Split Ratio': "70:15:15",
        'Best Epoch': best_epoch,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'Train Time (s)': end_train - start_train,
        'Test Time (s)': end_test - start_test,
        'Memory Usage (MB)': get_memory_usage()
    }

    return result, model  # ✅ Now we return the trained model

# === Run Training ===
print("\n🚀 Training CNN with 70:15:15 split")
result, model = train_evaluate()

# Save results
df_results = pd.DataFrame([result])
df_results.to_excel('cnn_final_results.xlsx', index=False, engine='openpyxl')
print("\n✅ Final results saved to cnn_final_results.xlsx")

# === SHAP Analysis ===
print("\n🔎 Running SHAP analysis...")

# Background: small subset of training data
background = torch.tensor(X_scaled[:50], dtype=torch.float32).unsqueeze(1)

# Test samples for explanation
test_samples = torch.tensor(X_scaled[:10], dtype=torch.float32).unsqueeze(1)

# Put model in eval mode
model.eval()

# GradientExplainer
explainer = shap.GradientExplainer(model, background)
shap_values = explainer.shap_values(test_samples)

print("SHAP values computed ✅")
print("Number of classes:", len(shap_values))
print("Shape for class 0 explanations:", shap_values[0].shape)


In [ ]:
for i, s in enumerate(shap_values):
    print(f"Class {i} SHAP shape:", np.array(s).shape)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample_idx = 0   # pick first sample
target_class = 0 # S1 = 0, S2 = 1, S3 = 2, S4 = 3

# Extract SHAP values for the chosen sample & class
shap_vals = shap_values[target_class][sample_idx, :, target_class]  # shape (2901,)

plt.figure(figsize=(12,5))
plt.plot(range(1, 2902), shap_vals, label=f"SHAP for class {le.classes_[target_class]}")
plt.xlabel("Raman Wavenumber Index")
plt.ylabel("SHAP Value")
plt.title(f"Feature Importance (SHAP) for {le.classes_[target_class]}")
plt.legend()
plt.show()

plt.show()


In [ ]:
print("Shape of X (features):", X.shape)
print("First 5 column names:", X.columns[:5].tolist())
print("Last 5 column names:", X.columns[-5:].tolist())


In [ ]:
# === Step 1: Load and preprocess ===
df = pd.read_csv('1.csv')
X = df.drop(columns=['Algae_Type', 'ID'])
y = df['Algae_Type']

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# ---------  Safety checks / fetch arrays ----------
X_raw = X.values if isinstance(X, pd.DataFrame) else np.array(X)
X_scaled_arr = np.array(X_scaled)

print("X_raw.shape:", X_raw.shape)
print("X_scaled.shape:", X_scaled_arr.shape)

# ---------  Mean / Std / SNR on RAW intensities ----------
mean_raw = np.mean(X_raw, axis=0)
std_raw  = np.std(X_raw, axis=0)
snr_raw  = mean_raw / (std_raw + 1e-9)

raman_shifts = np.arange(1, X_raw.shape[1] + 1)

# ---------  Peak detection on raw mean spectrum ----------
prominence_threshold = np.max(mean_raw) * 0.001
peaks, properties = find_peaks(mean_raw, prominence=prominence_threshold)
peaks_df = pd.DataFrame({
    "RamanShift": raman_shifts[peaks],
    "MeanIntensity": mean_raw[peaks],
    "Prominence": properties.get("prominences", np.full_like(peaks, np.nan)),
    "StdAcrossSamples": std_raw[peaks],
    "SNR_true": snr_raw[peaks]
}).sort_values("SNR_true", ascending=False).reset_index(drop=True)

print("Detected peaks (top 20 by SNR):")
print(peaks_df.head(20))

# ---------  SHAP extraction ----------
def compute_mean_abs_shap(shap_vals, target_class):
    if hasattr(shap_vals, "values"):
        arr = np.array(shap_vals.values)
    else:
        if isinstance(shap_vals, (list, tuple)):
            try:
                arr = np.stack([np.array(x) for x in shap_vals], axis=0)
            except Exception:
                arr = np.array(shap_vals)
        else:
            arr = np.array(shap_vals)
    arr = np.squeeze(arr)
    if arr.ndim == 3:
        return np.mean(np.abs(arr[:, :, target_class]), axis=0)
    elif arr.ndim == 2:
        return np.mean(np.abs(arr), axis=0)
    elif arr.ndim == 4:
        b, s, f, c = arr.shape
        arr2 = arr.reshape(b*s, f, c)
        return np.mean(np.abs(arr2[:, :, target_class]), axis=0)
    else:
        raise ValueError("Unexpected shap_values structure with ndim=", arr.ndim)

# Pick class
target_class = 0
mean_abs_shap = compute_mean_abs_shap(shap_values, target_class)
print("mean_abs_shap shape:", mean_abs_shap.shape)

# ---------  Plot customization ----------
plt.rcParams["font.family"] = "Book Antiqua"   # use Book Antiqua globally

# SRM University colors
colors = {
    "shap": "#C0C0C0",
    "spectrum": "#AB7100",
    "std": "#ADA66F",
    "shap_points": "#ffffff",
    "peaks": "#C3892B"
}

fig, ax_shap = plt.subplots(figsize=(12,6))

# Left axis: SHAP
ax_shap.plot(raman_shifts, mean_abs_shap, color=colors["shap"], lw=1.5,
             label=f"Mean |SHAP| ({le.classes_[target_class]})")
ax_shap.set_xlabel("Raman shift (cm$^{-1}$)", fontsize=14, color="black")
ax_shap.set_ylabel("Mean |SHAP| value", fontsize=14, color="black")
ax_shap.tick_params(axis='y', colors="black", labelsize=14)  # y-axis tick labels
ax_shap.tick_params(axis='x', colors="black", labelsize=14)  # x-axis tick labels

# Top SHAP peaks
top_shap_idx = np.argsort(mean_abs_shap)[-28:][::-1]
ax_shap.scatter(raman_shifts[top_shap_idx], mean_abs_shap[top_shap_idx],
                color=colors["shap_points"], s=30, edgecolor='#AB7100', zorder=5, label="Top SHAP")

# Right axis: raw spectrum
ax_spec = ax_shap.twinx()
ax_spec.plot(raman_shifts, mean_raw, color=colors["spectrum"], lw=1.3, label="Mean raw spectrum")
ax_spec.fill_between(raman_shifts, mean_raw - std_raw, mean_raw + std_raw,
                     color=colors["std"], alpha=0.5, label="SD")
ax_spec.set_ylabel("Intensity (raw a.u.)", fontsize=14, color="black")
ax_spec.tick_params(axis='y', colors="black", labelsize=14)  # y-axis tick labels

# Peaks as plain dots
ax_spec.scatter(peaks_df["RamanShift"], peaks_df["MeanIntensity"],
                color='#494623',  # color
                s=50,              # size of dots
                edgecolor='none',  # remove edge
                zorder=6,
                label="Detected peaks")

# --------- Legend inside plot ----------
handles, labels = [], []
for ax in [ax_shap, ax_spec]:
    h, l = ax.get_legend_handles_labels()
    handles += h
    labels += l
ax_shap.legend(handles, labels, loc="upper right", fontsize=14, frameon=True)

#plt.title(f"SHAP vs Raw Raman spectrum — {le.classes_[target_class]}", fontsize=16, color="black")
plt.tight_layout()
# ✅ Save BEFORE showing
plt.savefig("shap_S1.png", dpi=1200, transparent=True, bbox_inches='tight')
plt.show()


In [ ]:
# === Step 1: Load and preprocess ===
df = pd.read_csv('2.csv')
X = df.drop(columns=['Algae_Type', 'ID'])
y = df['Algae_Type']

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# ---------  Safety checks / fetch arrays ----------
X_raw = X.values if isinstance(X, pd.DataFrame) else np.array(X)
X_scaled_arr = np.array(X_scaled)

print("X_raw.shape:", X_raw.shape)
print("X_scaled.shape:", X_scaled_arr.shape)

# ---------  Mean / Std / SNR on RAW intensities ----------
mean_raw = np.mean(X_raw, axis=0)
std_raw  = np.std(X_raw, axis=0)
snr_raw  = mean_raw / (std_raw + 1e-9)

raman_shifts = np.arange(1, X_raw.shape[1] + 1)

# ---------  Peak detection on raw mean spectrum ----------
prominence_threshold = np.max(mean_raw) * 0.001
peaks, properties = find_peaks(mean_raw, prominence=prominence_threshold)
peaks_df = pd.DataFrame({
    "RamanShift": raman_shifts[peaks],
    "MeanIntensity": mean_raw[peaks],
    "Prominence": properties.get("prominences", np.full_like(peaks, np.nan)),
    "StdAcrossSamples": std_raw[peaks],
    "SNR_true": snr_raw[peaks]
}).sort_values("SNR_true", ascending=False).reset_index(drop=True)

print("Detected peaks (top 20 by SNR):")
print(peaks_df.head(20))

# ---------  SHAP extraction ----------
def compute_mean_abs_shap(shap_vals, target_class):
    if hasattr(shap_vals, "values"):
        arr = np.array(shap_vals.values)
    else:
        if isinstance(shap_vals, (list, tuple)):
            try:
                arr = np.stack([np.array(x) for x in shap_vals], axis=0)
            except Exception:
                arr = np.array(shap_vals)
        else:
            arr = np.array(shap_vals)
    arr = np.squeeze(arr)
    if arr.ndim == 3:
        return np.mean(np.abs(arr[:, :, target_class]), axis=0)
    elif arr.ndim == 2:
        return np.mean(np.abs(arr), axis=0)
    elif arr.ndim == 4:
        b, s, f, c = arr.shape
        arr2 = arr.reshape(b*s, f, c)
        return np.mean(np.abs(arr2[:, :, target_class]), axis=0)
    else:
        raise ValueError("Unexpected shap_values structure with ndim=", arr.ndim)

# Pick class
target_class = 1
mean_abs_shap = compute_mean_abs_shap(shap_values, target_class)
print("mean_abs_shap shape:", mean_abs_shap.shape)

# ---------  Plot customization ----------
plt.rcParams["font.family"] = "Book Antiqua"   # use Book Antiqua globally

# SRM University colors
colors = {
    "shap": "#C0C0C0",
    "spectrum": "#AB7100",
    "std": "#ADA66F",
    "shap_points": "#ffffff",
    "peaks": "#C3892B"
}

fig, ax_shap = plt.subplots(figsize=(12,6))

# Left axis: SHAP
ax_shap.plot(raman_shifts, mean_abs_shap, color=colors["shap"], lw=1.5,
             label=f"Mean |SHAP| ({le.classes_[target_class]})")
ax_shap.set_xlabel("Raman shift (cm$^{-1}$)", fontsize=14, color="black")

ax_shap.set_ylabel("Mean |SHAP| value", fontsize=14, color="black")
ax_shap.tick_params(axis='y', colors="black", labelsize=14)  # y-axis tick labels
ax_shap.tick_params(axis='x', colors="black", labelsize=14)  # x-axis tick labels

# Top SHAP peaks
top_shap_idx = np.argsort(mean_abs_shap)[-28:][::-1]
ax_shap.scatter(raman_shifts[top_shap_idx], mean_abs_shap[top_shap_idx],
                color=colors["shap_points"], s=30, edgecolor='#AB7100', zorder=5, label="Top SHAP")

# Right axis: raw spectrum
ax_spec = ax_shap.twinx()
ax_spec.plot(raman_shifts, mean_raw, color=colors["spectrum"], lw=1.3, label="Mean raw spectrum")
ax_spec.fill_between(raman_shifts, mean_raw - std_raw, mean_raw + std_raw,
                     color=colors["std"], alpha=0.5, label="SD")
ax_spec.set_ylabel("Intensity (raw a.u.)", fontsize=14, color="black")
ax_spec.tick_params(axis='y', colors="black", labelsize=14)  # y-axis tick labels

# Peaks as plain dots
ax_spec.scatter(peaks_df["RamanShift"], peaks_df["MeanIntensity"],
                color='#494623',  # color
                s=50,              # size of dots
                edgecolor='none',  # remove edge
                zorder=6,
                label="Detected peaks")

# --------- Legend inside plot ----------
handles, labels = [], []
for ax in [ax_shap, ax_spec]:
    h, l = ax.get_legend_handles_labels()
    handles += h
    labels += l
ax_shap.legend(handles, labels, loc="upper right", fontsize=14, frameon=True)

#plt.title(f"SHAP vs Raw Raman spectrum — {le.classes_[target_class]}", fontsize=16, color="black")
plt.tight_layout()

plt.savefig("shap_S2.png", dpi=1200, transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
# === Step 1: Load and preprocess ===
df = pd.read_csv('3.csv')
X = df.drop(columns=['Algae_Type', 'ID'])
y = df['Algae_Type']

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# ---------  Safety checks / fetch arrays ----------
X_raw = X.values if isinstance(X, pd.DataFrame) else np.array(X)
X_scaled_arr = np.array(X_scaled)

print("X_raw.shape:", X_raw.shape)
print("X_scaled.shape:", X_scaled_arr.shape)

# ---------  Mean / Std / SNR on RAW intensities ----------
mean_raw = np.mean(X_raw, axis=0)
std_raw  = np.std(X_raw, axis=0)
snr_raw  = mean_raw / (std_raw + 1e-9)

raman_shifts = np.arange(1, X_raw.shape[1] + 1)

# ---------  Peak detection on raw mean spectrum ----------
prominence_threshold = np.max(mean_raw) * 0.001
peaks, properties = find_peaks(mean_raw, prominence=prominence_threshold)
peaks_df = pd.DataFrame({
    "RamanShift": raman_shifts[peaks],
    "MeanIntensity": mean_raw[peaks],
    "Prominence": properties.get("prominences", np.full_like(peaks, np.nan)),
    "StdAcrossSamples": std_raw[peaks],
    "SNR_true": snr_raw[peaks]
}).sort_values("SNR_true", ascending=False).reset_index(drop=True)

print("Detected peaks (top 20 by SNR):")
print(peaks_df.head(20))

# ---------  SHAP extraction ----------
def compute_mean_abs_shap(shap_vals, target_class):
    if hasattr(shap_vals, "values"):
        arr = np.array(shap_vals.values)
    else:
        if isinstance(shap_vals, (list, tuple)):
            try:
                arr = np.stack([np.array(x) for x in shap_vals], axis=0)
            except Exception:
                arr = np.array(shap_vals)
        else:
            arr = np.array(shap_vals)
    arr = np.squeeze(arr)
    if arr.ndim == 3:
        return np.mean(np.abs(arr[:, :, target_class]), axis=0)
    elif arr.ndim == 2:
        return np.mean(np.abs(arr), axis=0)
    elif arr.ndim == 4:
        b, s, f, c = arr.shape
        arr2 = arr.reshape(b*s, f, c)
        return np.mean(np.abs(arr2[:, :, target_class]), axis=0)
    else:
        raise ValueError("Unexpected shap_values structure with ndim=", arr.ndim)

# Pick class
target_class = 2
mean_abs_shap = compute_mean_abs_shap(shap_values, target_class)
print("mean_abs_shap shape:", mean_abs_shap.shape)

# ---------  Plot customization ----------
plt.rcParams["font.family"] = "Book Antiqua"   # use Book Antiqua globally

# SRM University colors
colors = {
    "shap": "#C0C0C0",
    "spectrum": "#AB7100",
    "std": "#ADA66F",
    "shap_points": "#ffffff",
    "peaks": "#C3892B"
}

fig, ax_shap = plt.subplots(figsize=(12,6))

# Left axis: SHAP
ax_shap.plot(raman_shifts, mean_abs_shap, color=colors["shap"], lw=1.5,
             label=f"Mean |SHAP| ({le.classes_[target_class]})")
ax_shap.set_xlabel("Raman shift (cm$^{-1}$)", fontsize=14, color="black")

ax_shap.set_ylabel("Mean |SHAP| value", fontsize=14, color="black")
ax_shap.tick_params(axis='y', colors="black", labelsize=14)  # y-axis tick labels
ax_shap.tick_params(axis='x', colors="black", labelsize=14)  # x-axis tick labels

# Top SHAP peaks
top_shap_idx = np.argsort(mean_abs_shap)[-28:][::-1]
ax_shap.scatter(raman_shifts[top_shap_idx], mean_abs_shap[top_shap_idx],
                color=colors["shap_points"], s=30, edgecolor='#AB7100', zorder=5, label="Top SHAP")

# Right axis: raw spectrum
ax_spec = ax_shap.twinx()
ax_spec.plot(raman_shifts, mean_raw, color=colors["spectrum"], lw=1.3, label="Mean raw spectrum")
ax_spec.fill_between(raman_shifts, mean_raw - std_raw, mean_raw + std_raw,
                     color=colors["std"], alpha=0.5, label="SD")
ax_spec.set_ylabel("Intensity (raw a.u.)", fontsize=14, color="black")
ax_spec.tick_params(axis='y', colors="black", labelsize=14)  # y-axis tick labels

# Peaks as plain dots
ax_spec.scatter(peaks_df["RamanShift"], peaks_df["MeanIntensity"],
                color='#494623',  # color
                s=50,              # size of dots
                edgecolor='none',  # remove edge
                zorder=6,
                label="Detected peaks")

# --------- Legend inside plot ----------
handles, labels = [], []
for ax in [ax_shap, ax_spec]:
    h, l = ax.get_legend_handles_labels()
    handles += h
    labels += l
ax_shap.legend(handles, labels, loc="upper right", fontsize=14, frameon=True)

#plt.title(f"SHAP vs Raw Raman spectrum — {le.classes_[target_class]}", fontsize=16, color="black")
plt.tight_layout()

plt.savefig("shap_S3.png", dpi=1200, transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
# === Step 1: Load and preprocess ===
df = pd.read_csv('4.csv')
X = df.drop(columns=['Algae_Type', 'ID'])
y = df['Algae_Type']

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# ---------  Safety checks / fetch arrays ----------
X_raw = X.values if isinstance(X, pd.DataFrame) else np.array(X)
X_scaled_arr = np.array(X_scaled)

print("X_raw.shape:", X_raw.shape)
print("X_scaled.shape:", X_scaled_arr.shape)

# ---------  Mean / Std / SNR on RAW intensities ----------
mean_raw = np.mean(X_raw, axis=0)
std_raw  = np.std(X_raw, axis=0)
snr_raw  = mean_raw / (std_raw + 1e-9)

raman_shifts = np.arange(1, X_raw.shape[1] + 1)

# ---------  Peak detection on raw mean spectrum ----------
prominence_threshold = np.max(mean_raw) * 0.001
peaks, properties = find_peaks(mean_raw, prominence=prominence_threshold)
peaks_df = pd.DataFrame({
    "RamanShift": raman_shifts[peaks],
    "MeanIntensity": mean_raw[peaks],
    "Prominence": properties.get("prominences", np.full_like(peaks, np.nan)),
    "StdAcrossSamples": std_raw[peaks],
    "SNR_true": snr_raw[peaks]
}).sort_values("SNR_true", ascending=False).reset_index(drop=True)

print("Detected peaks (top 20 by SNR):")
print(peaks_df.head(20))

# ---------  SHAP extraction ----------
def compute_mean_abs_shap(shap_vals, target_class):
    if hasattr(shap_vals, "values"):
        arr = np.array(shap_vals.values)
    else:
        if isinstance(shap_vals, (list, tuple)):
            try:
                arr = np.stack([np.array(x) for x in shap_vals], axis=0)
            except Exception:
                arr = np.array(shap_vals)
        else:
            arr = np.array(shap_vals)
    arr = np.squeeze(arr)
    if arr.ndim == 3:
        return np.mean(np.abs(arr[:, :, target_class]), axis=0)
    elif arr.ndim == 2:
        return np.mean(np.abs(arr), axis=0)
    elif arr.ndim == 4:
        b, s, f, c = arr.shape
        arr2 = arr.reshape(b*s, f, c)
        return np.mean(np.abs(arr2[:, :, target_class]), axis=0)
    else:
        raise ValueError("Unexpected shap_values structure with ndim=", arr.ndim)

# Pick class
target_class = 3
mean_abs_shap = compute_mean_abs_shap(shap_values, target_class)
print("mean_abs_shap shape:", mean_abs_shap.shape)

# ---------  Plot customization ----------
plt.rcParams["font.family"] = "Book Antiqua"   # use Book Antiqua globally

# SRM University colors
colors = {
    "shap": "#C0C0C0",
    "spectrum": "#AB7100",
    "std": "#ADA66F",
    "shap_points": "#ffffff",
    "peaks": "#C3892B"
}

fig, ax_shap = plt.subplots(figsize=(12,6))

# Left axis: SHAP
ax_shap.plot(raman_shifts, mean_abs_shap, color=colors["shap"], lw=1.5,
             label=f"Mean |SHAP| ({le.classes_[target_class]})")
ax_shap.set_xlabel("Raman shift (cm$^{-1}$)", fontsize=14, color="black")

ax_shap.set_ylabel("Mean |SHAP| value", fontsize=14, color="black")
ax_shap.tick_params(axis='y', colors="black", labelsize=14)  # y-axis tick labels
ax_shap.tick_params(axis='x', colors="black", labelsize=14)  # x-axis tick labels

# Top SHAP peaks
top_shap_idx = np.argsort(mean_abs_shap)[-28:][::-1]
ax_shap.scatter(raman_shifts[top_shap_idx], mean_abs_shap[top_shap_idx],
                color=colors["shap_points"], s=30, edgecolor='#AB7100', zorder=5, label="Top SHAP")

# Right axis: raw spectrum
ax_spec = ax_shap.twinx()
ax_spec.plot(raman_shifts, mean_raw, color=colors["spectrum"], lw=1.3, label="Mean raw spectrum")
ax_spec.fill_between(raman_shifts, mean_raw - std_raw, mean_raw + std_raw,
                     color=colors["std"], alpha=0.5, label="SD")
ax_spec.set_ylabel("Intensity (raw a.u.)", fontsize=14, color="black")
ax_spec.tick_params(axis='y', colors="black", labelsize=14)  # y-axis tick labels

# Peaks as plain dots
ax_spec.scatter(peaks_df["RamanShift"], peaks_df["MeanIntensity"],
                color='#494623',  # color
                s=50,              # size of dots
                edgecolor='none',  # remove edge
                zorder=6,
                label="Detected peaks")

# --------- Legend inside plot ----------
handles, labels = [], []
for ax in [ax_shap, ax_spec]:
    h, l = ax.get_legend_handles_labels()
    handles += h
    labels += l
ax_shap.legend(handles, labels, loc="upper right", fontsize=14, frameon=True)

#plt.title(f"SHAP vs Raw Raman spectrum — {le.classes_[target_class]}", fontsize=16, color="black")
plt.tight_layout()

plt.savefig("shap_S4.png", dpi=1200, transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
# === PCA Visualization for Raman Data (Raw vs Noise-Cancelled) ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from scipy.signal import savgol_filter

# --- SNV normalization ---
def snv(input_data):
    return (input_data - np.mean(input_data, axis=1, keepdims=True)) / np.std(input_data, axis=1, keepdims=True)

# === Step 1: Load dataset ===
df = pd.read_csv("Dataset _Raman.csv")
X = df.drop(columns=['Algae_Type', 'ID'])
y = df['Algae_Type']

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Species colors
species_colors = {
    "S1": "#Ab7100",  # blue
    "S2": "#666666",  # orange
    "S3": "#C1a700",  # green
    "S4": "#C3892B"   # red
}
edge_colors = {

    "S1": "#ffffff",  # blue
    "S2": "#ffffff",  # orange
    "S3": "#ffffff",  # green
    "S4": "#ffffff"   # red
}
# === Step 2: Build dataset versions ===
X_raw = X.values

# Noise-cancelled (Savitzky-Golay + SNV + scaling)
X_sg = savgol_filter(X_raw, window_length=25, polyorder=2, deriv=1, axis=1)
X_snv_sg = snv(X_sg)
X_noise_cancel = StandardScaler().fit_transform(X_snv_sg)

datasets = {
    "Raw": X_raw,
    "Preprocessed & Noise-Cancelled (SG+SNV+Scaling)": X_noise_cancel
}


In [ ]:
# === Step 3: PCA Visualization (95% explained variance) ===
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, X_version) in zip(axes, datasets.items()):
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_version, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
    )
   
    # PCA fit (retain 95% variance)
    pca = PCA(n_components=0.95)
    X_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)
   
    # Plot 2D scatter (first 2 PCs)
    for label in np.unique(y_test):
        species_name = le.classes_[label]
        ax.scatter(
            X_test_pca[y_test == label, 0],
            X_test_pca[y_test == label, 1],
            color=species_colors[species_name],
            s=30,
            alpha=0.8,
            marker='o',
            edgecolor=edge_colors[species_name],  # or "none"
            label=species_name
        )
    
    ax.set_title(f"{name}\nPCs: {pca.n_components_}, Expl.Var: {pca.explained_variance_ratio_.sum():.2f}",
                 fontsize=14)
    ax.set_xlabel("PC1", fontsize=14)
    ax.set_ylabel("PC2", fontsize=14)
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)

# Global legend at bottom
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels,
           loc="lower center",
           bbox_to_anchor=(0.5, -0.09),
           fontsize=14,
           ncol=len(labels),
           frameon=True)

#plt.suptitle("PCA Visualization of Raman Data", fontsize=16)

# Make space for suptitle
plt.tight_layout(rect=[0, 0, 1, 0.95])

# Save *before* showing
plt.savefig("PCA_Vis.png", dpi=1200, transparent=True, bbox_inches='tight')
plt.show()
